# 01 Process new data

- This process consists of aggregation all new data extracted daily from website https://stats.tennismylife.org/tennis-match-database 
- Only new items will be append to the final dataset

## Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import os

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

True

## Init spark

In [4]:
try:
    spark = SparkSession.builder.appName("raw_pipeline").getOrCreate()
except Exception as e:
    print(e)

## Load database

In [21]:
tb_atp_matches = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "raw.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

In [22]:
tb_ongoing_tourneys = spark.read.format("csv").option("header", "true").load(r"../../data/raw/tb_ongoing_tourneys.csv")

## Append new data

In [23]:
df = tb_atp_matches.unionByName(tb_ongoing_tourneys, allowMissingColumns=True)

## Save dataframe

### Local

In [24]:
df.toPandas().to_csv(
    r"../../data/raw/tb_atp_matches.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

### Supabase

In [25]:
(
tb_ongoing_tourneys.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "raw.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("append")
    .save()
)